[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/pytorch-lab/blob/main/01_pytorch_basics.ipynb)

# 01 — Основы PyTorch: тензоры и autograd

**Цель.** Освоить основную структуру данных PyTorch, выполнить базовые операции и увидеть, как работает автоматическое дифференцирование.

**Результаты обучения:**
- создавать тензоры разными способами
- выполнять арифметику, broadcasting и матричное умножение
- изменять форму тензоров и выбирать элементы по индексам
- запускать autograd и проверять производную аналитически
- переносить тензоры на GPU, если он доступен

## Источники
- [Heaton: PyTorch numerical](https://github.com/jeffheaton/app_deep_learning/blob/main/t81_558_class_02_1_pytorch_numerical.ipynb)
- [Heaton: Beyond CPU](https://github.com/jeffheaton/app_deep_learning/blob/main/t81_558_class_02_5_beyond_cpu.ipynb)
- [Lapan: Chapter03/01_modules.py](https://github.com/PacktPublishing/Deep-Reinforcement-Learning-Hands-On/blob/master/Chapter03/01_modules.py)

Импортируем основные библиотеки. Для воспроизводимости фиксируем случайные зерна PyTorch и NumPy.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

Создадим несколько тензоров: заполненные нулями, единицами, заданными значениями и случайными числами. Обратите внимание на форму и тип данных.

In [ ]:
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
values = torch.tensor([1.0, 2.0, 3.0])
random_tensor = torch.rand(2, 3)

for name, tensor in {
    "zeros": zeros,
    "ones": ones,
    "values": values,
    "random_tensor": random_tensor,
}.items():
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")
    print(tensor)

Тензоры поддерживают обычную арифметику. Broadcasting позволяет складывать массивы совместимых форм без ручного копирования данных.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]])

print("a + b =")
print(a + b)

print("Elementwise multiplication:")
print(a * b)

print("Matrix multiplication with @:")
print(a @ b)

print("Matrix multiplication with torch.matmul:")
print(torch.matmul(a, b))

column = torch.arange(3).reshape(3, 1)
row = torch.arange(4).reshape(1, 4)
print("Broadcasting result shape:", (column + row).shape)
print(column + row)

Форму тензора можно менять через `view` и `reshape`. Срезы работают похоже на NumPy: можно выбирать строки, столбцы и диапазоны.

In [ ]:
base = torch.arange(12)
matrix_view = base.view(3, 4)
matrix_reshape = base.reshape(2, 6)

print("Original:", base)
print("View 3x4:")
print(matrix_view)
print("Reshape 2x6:")
print(matrix_reshape)

print("First row:", matrix_view[0])
print("Second column:", matrix_view[:, 1])
print("Bottom-right block:")
print(matrix_view[1:, 2:])

На CPU тензоры PyTorch и массивы NumPy могут разделять одну память. Изменение одного объекта видно в другом.

In [ ]:
np_array = np.array([1.0, 2.0, 3.0], dtype=np.float32)
torch_from_np = torch.from_numpy(np_array)

torch_from_np[0] = 99.0
print("NumPy after tensor update:", np_array)

back_to_np = torch_from_np.numpy()
back_to_np[1] = -5.0
print("Tensor after NumPy update:", torch_from_np)

Autograd строит граф вычислений и автоматически считает производные. Для функции `y = x² + 3x + 1` производная равна `2x + 3`; при `x = 2` ожидаем `7`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x + 1
y.backward()

print("y:", y.item())
print("dy/dx from autograd:", x.grad.item())
print("Analytical dy/dx:", 2 * x.item() + 3)

Autograd работает и для цепочек операций с несколькими переменными. Ниже считаем градиенты расстояния `sqrt(a² + b²)`.

In [ ]:
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)
z = (a**2 + b**2).sqrt()
z.backward()

print("z:", z.item())
print("dz/da:", a.grad.item())
print("dz/db:", b.grad.item())

Проверим доступность GPU. Если CUDA есть, перенесем тензор на устройство и сравним время большого матричного умножения на CPU и GPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

sample = torch.rand(3, 3)
print("Sample tensor device before:", sample.device)
print("Sample tensor device after:", sample.to(device).device)

size = 2000
cpu_a = torch.rand(size, size)
cpu_b = torch.rand(size, size)

start = time.perf_counter()
_ = cpu_a @ cpu_b
cpu_time = time.perf_counter() - start
print(f"CPU matmul time: {cpu_time:.4f} seconds")

if torch.cuda.is_available():
    gpu_a = cpu_a.to("cuda")
    gpu_b = cpu_b.to("cuda")
    torch.cuda.synchronize()
    start = time.perf_counter()
    _ = gpu_a @ gpu_b
    torch.cuda.synchronize()
    gpu_time = time.perf_counter() - start
    print(f"GPU matmul time: {gpu_time:.4f} seconds")
    print(f"Speedup: {cpu_time / gpu_time:.2f}x")
else:
    print("GPU is not available in this runtime.")

> Материал основан на курсе Jeff Heaton T81-558 и книге Maxim Lapan 'Deep Reinforcement Learning Hands-On', глава 3. Адаптировано для учебных целей.